# Cross-Construct Comparison Plots

Regenerates the three comparison box-with-swarm plots and summary statistics
from the **already-saved CSV data** produced by `run_burst_analysis.py`.

No need to re-run the pipeline or mount the data drive.

**Plots generated** (PNG + SVG via `save_figure()`):
1. Observed ON Episode Duration (trajectory-level median)
2. Observed OFF Episode Duration (trajectory-level median)
3. Fraction of Observed Time ON

**Statistical notes:**
- ON/OFF duration plots use **trajectory-level medians** (one value per trajectory), not pooled events
- Initial OFF dwells are **excluded** (left-censored)
- Terminal OFF dwells are **excluded**
- All pairwise Mann-Whitney tests include **optional Benjamini-Hochberg FDR correction**

**Data source:** `results_snr_3/{construct}/trajectory_summary.csv` and `event_table.csv`

In [ ]:
"""Cell 1: Setup — imports, paths, construct order, publication style."""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ── Robust path setup ──
# This notebook lives in time_courses/.  Walk up to find the repo root
# (.git marker), then add both repo root and time_courses/ to sys.path
# so imports work regardless of where Jupyter was launched from.
_nb_dir = Path.cwd()  # typically time_courses/ if launched from there
repo_root = _nb_dir
while repo_root != repo_root.parent and not (repo_root / ".git").exists():
    repo_root = repo_root.parent
if not (repo_root / ".git").exists():
    raise RuntimeError(
        f"Could not find repo root (.git) walking up from {_nb_dir}. "
        "Please run this notebook from within the cof_paper repository."
    )
_tc_dir = repo_root / "time_courses"
for p in [str(repo_root), str(_tc_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from plotting import (
    box_with_points,
    save_figure,
    set_publication_style,
)

set_publication_style()

# ═══════════════════════════════════════════════════════════════════
#  CONFIGURATION — edit this section for different pipeline runs
# ═══════════════════════════════════════════════════════════════════

# Which results directory to read.  Change this for a different SNR run.
RESULTS_DIR = repo_root / "time_courses" / "results_snr_3"

# Explicit construct order — must match the pipeline's processing order.
# DO NOT rely on directory sorting (4sf-Xbp1 sorts before 4uv).
CONSTRUCT_ORDER = ["4sf", "4uv", "4sf-Xbp1", "Xbp1-4sf"]

# Cell counts from the pipeline run.  These are QC-passing cell counts
# (unique FOVs with at least one surviving trajectory), NOT folder counts.
# They come from the existing comparison plots and cannot be reproduced
# from CSVs alone without re-running the full pipeline with ParticleOrigin.
# ⚠️  Valid ONLY for the results_snr_3 run.  Update if RESULTS_DIR changes.
N_CELLS = {
    "4sf": 12,
    "4uv": 21,
    "4sf-Xbp1": 15,
    "Xbp1-4sf": 16,
}

# Plot settings — mirror PLOT_PARAMS from config.yaml / run_burst_analysis.py
FIGSIZE = (5.5, 5.5)
PLOT_DPI = 300
USE_BH_FDR = False  # Set to True to apply Benjamini-Hochberg FDR multiple-testing correction

# Output directory
COMP_DIR = RESULTS_DIR / "comparison"
COMP_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root:   {repo_root}")
print(f"Results dir: {RESULTS_DIR}")
print(f"Output dir:  {COMP_DIR}")
print(f"Constructs:  {CONSTRUCT_ORDER}")

In [ ]:
"""Cell 2: Load trajectory_summary.csv and event_table.csv per construct.

ON/OFF durations use trajectory-level medians (one value per trajectory)
to avoid pseudoreplication from pooled events.
Initial and terminal OFF dwells are excluded (censored).
"""

# Data dictionaries — same structure as run_cross_construct_comparison()
frac_on_vals = {}    # short_name -> array of fraction_time_on (per trajectory)
burst_durs = {}      # short_name -> array of trajectory-level median ON durations
dwell_durs = {}      # short_name -> array of trajectory-level median OFF durations
n_trajectories = {}  # short_name -> int
n_on_events = {}     # short_name -> int (burst events passing duration filter)
n_off_events = {}    # short_name -> int (dwell events, non-initial, non-terminal)

# Accumulator for pairwise stats across all 3 plots (Cells 4–6).
# Initialized here so cells can be re-run independently.
stats_rows = []

for short in CONSTRUCT_ORDER:
    construct_dir = RESULTS_DIR / short
    ts_path = construct_dir / "trajectory_summary.csv"
    et_path = construct_dir / "event_table.csv"

    if not ts_path.exists() or not et_path.exists():
        print(f"  WARNING: Missing data for {short}, skipping")
        continue

    ts = pd.read_csv(ts_path)
    et = pd.read_csv(et_path)

    # Fraction ON: from trajectory summary (already per-trajectory)
    frac_on_vals[short] = ts["fraction_time_on"].values
    n_trajectories[short] = len(ts)

    # Burst (ON) durations: events that pass the duration filter
    bursts = et[(et["event_type"] == "burst") & et["passes_duration_filter"]]
    n_on_events[short] = len(bursts)

    # Dwell (OFF) durations: exclude initial (left-censored) and terminal
    dwells = et[
        (et["event_type"] == "dwell")
        & ~et["is_terminal_event"]
        & ~et["is_initial_dwell"]
    ]
    n_off_events[short] = len(dwells)

    # Trajectory-level median durations (one value per trajectory)
    burst_durs[short] = (
        bursts.groupby("trajectory_id")["duration_minutes"].median().values
        if not bursts.empty else np.array([])
    )
    dwell_durs[short] = (
        dwells.groupby("trajectory_id")["duration_minutes"].median().values
        if not dwells.empty else np.array([])
    )

    print(
        f"  {short:12s} | {n_trajectories[short]:4d} traj | "
        f"{n_on_events[short]:4d} ON events | {n_off_events[short]:4d} OFF events | "
        f"{len(burst_durs[short]):4d} traj w/ ON | {len(dwell_durs[short]):4d} traj w/ OFF"
    )

print(f"\nLoaded {len(frac_on_vals)} constructs")

In [ ]:
"""Cell 3: Helper — build multi-line x-axis labels."""


def build_xlabels(constructs, event_counts=None):
    """Build multi-line x-axis labels: name / cells / traj / (events)."""
    labels = []
    for c in constructs:
        parts = [
            c,
            f"{N_CELLS.get(c, 0)} cells",
            f"{n_trajectories.get(c, 0)} traj",
        ]
        if event_counts is not None:
            parts.append(f"({event_counts.get(c, 0)} events)")
        labels.append("\n".join(parts))
    return labels

In [ ]:
"""Cell 4: Observed ON Episode Duration comparison (saves PNG + SVG)."""

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, facecolor="white")
data = [burst_durs.get(c, []) for c in CONSTRUCT_ORDER]
xlabels_on = build_xlabels(CONSTRUCT_ORDER, n_on_events)

stats = box_with_points(
    ax,
    data,
    CONSTRUCT_ORDER,
    ylabel="Observed ON Episode Duration (min)",
    title="Observed ON Episode Duration (traj. median)",
    xlabels=xlabels_on,
    show_stats=True,
    only_significant=True,
    max_percentile_significance=99.5,
    use_bh_fdr=USE_BH_FDR,
)
for row in stats:
    stats_rows.append({"metric": "on_duration_minutes", **row})
fig.tight_layout()
save_figure(fig, COMP_DIR / "burst_duration_comparison", PLOT_DPI)

In [ ]:
"""Cell 5: Observed OFF Episode Duration comparison (saves PNG + SVG)."""

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, facecolor="white")
data = [dwell_durs.get(c, []) for c in CONSTRUCT_ORDER]
xlabels_off = build_xlabels(CONSTRUCT_ORDER, n_off_events)

stats = box_with_points(
    ax,
    data,
    CONSTRUCT_ORDER,
    ylabel="Observed OFF Episode Duration (min)",
    title="Observed OFF Episode Duration (traj. median)",
    xlabels=xlabels_off,
    show_stats=True,
    only_significant=True,
    max_percentile_significance=99.5,
    use_bh_fdr=USE_BH_FDR,
)
for row in stats:
    stats_rows.append({"metric": "off_duration_minutes", **row})
fig.tight_layout()
save_figure(fig, COMP_DIR / "dwell_duration_comparison", PLOT_DPI)

In [ ]:
"""Cell 6: Fraction of Observed Time ON comparison (saves PNG + SVG)."""

fig, ax = plt.subplots(1, 1, figsize=FIGSIZE, facecolor="white")
data = [frac_on_vals.get(c, []) for c in CONSTRUCT_ORDER]
xlabels_frac = build_xlabels(CONSTRUCT_ORDER)  # no event count for fraction

stats = box_with_points(
    ax,
    data,
    CONSTRUCT_ORDER,
    ylabel="Fraction of Observed Time ON",
    title="Fraction of Observed Time ON",
    ylim=(-0.05, 1.05),
    xlabels=xlabels_frac,
    show_stats=True,
    only_significant=True,
    max_percentile_significance=99.5,
    use_bh_fdr=USE_BH_FDR,
)
for row in stats:
    stats_rows.append({"metric": "fraction_time_on", **row})
fig.tight_layout()
save_figure(fig, COMP_DIR / "fraction_on_comparison", PLOT_DPI)

In [ ]:
"""Cell 7: Summary table and pairwise statistics.

All pairwise Mann-Whitney tests include optional Benjamini-Hochberg FDR correction.
Significance stars reflect chosen correction mode.
ON/OFF duration summary statistics are computed on trajectory-level medians.
"""

# ── Pairwise Mann-Whitney stats (accumulated from Cells 4–6) ──
stats_df = pd.DataFrame(stats_rows)
stats_path = COMP_DIR / "pairwise_mannwhitney_stats.csv"
stats_df.to_csv(stats_path, index=False)
print(f"Pairwise statistics saved to {stats_path}")
display(stats_df)

# ── Summary table ──
# ON/OFF duration values are trajectory-level medians, so summary stats
# here are "mean/median of per-trajectory median durations."
summary_rows = []
for short in CONSTRUCT_ORDER:
    bd = burst_durs.get(short, np.array([]))
    dd = dwell_durs.get(short, np.array([]))
    fo = frac_on_vals.get(short, np.array([]))

    summary_rows.append({
        "short_name": short,
        "n_cells": N_CELLS.get(short, 0),
        "n_trajectories": n_trajectories.get(short, 0),
        "n_on_events": n_on_events.get(short, 0),
        "n_off_events": n_off_events.get(short, 0),
        "n_traj_with_on": len(bd),
        "n_traj_with_off": len(dd),
        "mean_on_dur_min": round(float(np.mean(bd)), 3) if len(bd) > 0 else np.nan,
        "median_on_dur_min": round(float(np.median(bd)), 3) if len(bd) > 0 else np.nan,
        "std_on_dur_min": round(float(np.std(bd, ddof=1)), 3) if len(bd) > 1 else np.nan,
        "sem_on_dur_min": round(float(np.std(bd, ddof=1) / np.sqrt(len(bd))), 3) if len(bd) > 1 else np.nan,
        "mean_off_dur_min": round(float(np.mean(dd)), 3) if len(dd) > 0 else np.nan,
        "median_off_dur_min": round(float(np.median(dd)), 3) if len(dd) > 0 else np.nan,
        "std_off_dur_min": round(float(np.std(dd, ddof=1)), 3) if len(dd) > 1 else np.nan,
        "sem_off_dur_min": round(float(np.std(dd, ddof=1) / np.sqrt(len(dd))), 3) if len(dd) > 1 else np.nan,
        "mean_fraction_on": round(float(np.mean(fo)), 4) if len(fo) > 0 else np.nan,
        "median_fraction_on": round(float(np.median(fo)), 4) if len(fo) > 0 else np.nan,
        "std_fraction_on": round(float(np.std(fo, ddof=1)), 4) if len(fo) > 1 else np.nan,
        "sem_fraction_on": round(float(np.std(fo, ddof=1) / np.sqrt(len(fo))), 4) if len(fo) > 1 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = COMP_DIR / "summary_table.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\nSummary table saved to {summary_path}")
display(summary_df)